# 05 Fill Additive Marking — local_3x3 H=4 bottleneck CNN

Version 18 experiment.

This notebook is copied conceptually from `05-fill-additive-marking-local-3x3-17.ipynb`, but replaces the brittle hybrid rule-selection path with a direct learned local 3×3 function:

```text
Input  (1×10×30×30)
Conv1  3×3, 10 → 4, padding=1
ReLU
Conv2  1×1, 4 → 10
Output (1×10×30×30)
```

The target group is the 8 `fill_enclosed_regions` tasks marked `local_3x3_consistent`.

Important safety gates:

1. Every selected task must save an ONNX model.
2. Every saved model must be exact-match correct on visible `train + test + arc-gen` examples.
3. The notebook raises an assertion if `saved_count != 8` or if any visible validation has `wrong > 0`.

If H=4 cannot exactly fit a task, the notebook records the failure and stops instead of silently creating a weak submission.


In [ ]:
# Common helpers: paths, data loading, ONNX runtime validation, model reporting.

import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


BATCH, CH, GRID_H, GRID_W = 1, 10, 30, 30
FAMILY = "fill_enclosed_regions"
MODEL_VERSION = "fill-additive-local3x3-bottleneck-h4-v0.18"
HIDDEN_CHANNELS = 4


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
        return
    except Exception:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


# Kaggle usually has torch. ONNX/ORT may need installation, as in previous notebooks.
ensure_package("onnx")
ensure_package("onnxruntime")
try:
    import torch
    import torch.nn as nn
except Exception as exc:
    raise ImportError(
        "PyTorch is required for the H=4 bottleneck training/export experiment. "
        "Kaggle Python images normally include torch."
    ) from exc

import onnx
import onnxruntime as ort


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        root = Path("/kaggle/working")
        data_dir = kaggle_dir
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission" / FAMILY
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    candidates = [
        Path(path),
        Path("task_groups/task_type_map.csv"),
        Path("Co_Kaggle/g3/task_groups/task_type_map.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    """Return 1×10×30×30 float tensor with all-zero channels outside the true grid."""
    arr = np.zeros((BATCH, CH, GRID_H, GRID_W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < GRID_H and 0 <= c < GRID_W and 0 <= int(color) < CH:
                arr[0, int(color), r, c] = 1.0
    return arr


def examples_to_arrays(examples):
    xs = np.concatenate([grid_to_tensor(ex["input"]) for ex in examples], axis=0)
    ys = np.concatenate([grid_to_tensor(ex["output"]) for ex in examples], axis=0)
    return xs.astype(np.float32), ys.astype(np.float32)


def run_model(model_or_path, input_grid):
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    return {"right": right, "wrong": wrong, "total": right + wrong, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task):
    rows = {}
    for split, examples in {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }.items():
        right = 0
        wrong = 0
        for ex in examples:
            expected = grid_to_tensor(ex["output"])
            actual = run_model(model_or_path, ex["input"])
            if np.array_equal(actual, expected):
                right += 1
            else:
                wrong += 1
        total = right + wrong
        rows[split] = {
            "right": right,
            "wrong": wrong,
            "total": total,
            "accuracy": right / total if total else None,
        }
    visible = visible_validation_summary(model_or_path, task)
    rows["visible_all"] = {
        "right": visible["right"],
        "wrong": visible["wrong"],
        "total": visible["total"],
        "accuracy": visible["right"] / visible["total"] if visible["total"] else None,
    }
    return rows


def count_model_params(model_or_path):
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    return int(params)


def approximate_memory_from_model_shapes(model_or_path):
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    total = 0
    for value in list(inferred.graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 4
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    path = Path(model_or_path)
    model = onnx.load(str(path))
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 4
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(path),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(path),
        "file_size_bytes": path.stat().st_size,
        "profile_trace_path": trace_path,
    }


def model_architecture_summary(model_or_path):
    path = Path(model_or_path)
    model = onnx.load(str(path))
    op_counts = Counter(node.op_type for node in model.graph.node)
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "params": count_model_params(path),
        "file_size_bytes": path.stat().st_size,
    }


def model_report(model_or_path, task):
    examples = all_examples(task)
    sample_input_grid = examples[0]["input"]
    return {
        "architecture": model_architecture_summary(model_or_path),
        "memory_profile": runtime_memory_profile(model_or_path, sample_input_grid),
        "performance": split_validation_summary(model_or_path, task),
    }


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path


DATA_DIR, OUT_DIR = default_paths()
print("DATA_DIR =", DATA_DIR)
print("OUT_DIR =", OUT_DIR)
print("MODEL_VERSION =", MODEL_VERSION)
print("HIDDEN_CHANNELS =", HIDDEN_CHANNELS)
print("torch =", torch.__version__)
print("onnx =", onnx.__version__)
print("onnxruntime =", ort.__version__)


In [ ]:
# Select only the 8 local_3x3 fill/additive marking tasks.

task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()
local_3x3_df = family_df[family_df.candidate_flags.str.contains("local_3x3_consistent", na=False)].copy()
task_ids = local_3x3_df["task_id"].tolist()

print("family:", FAMILY)
print("family tasks:", len(family_df))
print("selected local_3x3 tasks:", len(task_ids))
display(local_3x3_df[[
    "task_id",
    "n_train",
    "n_test",
    "n_arc_gen",
    "shape_relation",
    "input_shape_modes",
    "output_shape_modes",
    "new_output_color_list",
    "local_3x3_score",
    "local_3x3_conflicts",
    "candidate_flags",
]])

assert len(task_ids) == 8, f"Expected 8 local_3x3 tasks, got {len(task_ids)}"
assert (local_3x3_df["local_3x3_score"].astype(float) == 1.0).all()
assert (local_3x3_df["local_3x3_conflicts"].astype(int) == 0).all()


In [ ]:
# H=4 bottleneck CNN training/export.

class BottleneckLocal3x3(nn.Module):
    def __init__(self, hidden_channels=HIDDEN_CHANNELS):
        super().__init__()
        self.conv1 = nn.Conv2d(CH, hidden_channels, kernel_size=3, padding=1, bias=True)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(hidden_channels, CH, kernel_size=1, padding=0, bias=True)

    def forward(self, x):
        return self.conv2(self.relu(self.conv1(x)))


def exact_tensor_stats_from_logits(logits, target):
    pred = (logits > 0).to(target.dtype)
    per_example = (pred == target).flatten(1).all(dim=1)
    right = int(per_example.sum().item())
    total = int(target.shape[0])
    wrong = total - right
    return right, wrong, total


def train_h4_bottleneck_for_task(
    task,
    *,
    task_id,
    hidden_channels=HIDDEN_CHANNELS,
    max_epochs=3500,
    check_every=50,
    seeds=(0, 1, 2, 3, 4),
    lr=0.03,
    weight_decay=0.0,
    positive_weight=3.0,
):
    """Train H=4 local 3×3 bottleneck model until exact visible match.

    Uses BCEWithLogitsLoss rather than softmax CE so padded cells outside the true
    grid can correctly target all-zero output channels.
    """
    examples = all_examples(task)
    x_np, y_np = examples_to_arrays(examples)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    x = torch.from_numpy(x_np).to(device)
    y = torch.from_numpy(y_np).to(device)

    # Positive pixels are sparse; a small pos_weight helps push true channels above zero.
    pos_weight_tensor = torch.full((CH, 1, 1), float(positive_weight), device=device)

    attempts = []
    best_state = None
    best_wrong = 10**9
    best_loss = float("inf")

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        model = BottleneckLocal3x3(hidden_channels=hidden_channels).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

        seed_best = {"seed": seed, "epoch": None, "wrong": 10**9, "loss": float("inf")}
        start = time.time()

        for epoch in range(1, max_epochs + 1):
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            if epoch == 1 or epoch % check_every == 0 or epoch == max_epochs:
                with torch.no_grad():
                    logits = model(x)
                    right, wrong, total = exact_tensor_stats_from_logits(logits, y)
                    loss_value = float(criterion(logits, y).item())

                if wrong < seed_best["wrong"] or (wrong == seed_best["wrong"] and loss_value < seed_best["loss"]):
                    seed_best = {
                        "seed": seed,
                        "epoch": epoch,
                        "right": right,
                        "wrong": wrong,
                        "total": total,
                        "loss": loss_value,
                        "elapsed_sec": time.time() - start,
                    }

                if wrong < best_wrong or (wrong == best_wrong and loss_value < best_loss):
                    best_wrong = wrong
                    best_loss = loss_value
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

                if wrong == 0:
                    attempts.append(seed_best)
                    model_cpu = BottleneckLocal3x3(hidden_channels=hidden_channels)
                    model_cpu.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()})
                    model_cpu.eval()
                    return model_cpu, {
                        "ok": True,
                        "task_id": task_id,
                        "trainer": "bottleneck_local3x3_h4_cnn",
                        "model_version": MODEL_VERSION,
                        "hidden_channels": hidden_channels,
                        "seed": seed,
                        "epoch": epoch,
                        "train_examples": len(task.get("train", [])),
                        "test_examples": len(task.get("test", [])),
                        "arc_gen_examples": len(task.get("arc-gen", [])),
                        "visible_examples": len(examples),
                        "visible_right": right,
                        "visible_wrong": wrong,
                        "visible_total": total,
                        "loss": loss_value,
                        "attempts": attempts,
                    }

        attempts.append(seed_best)
        print(f"{task_id} seed={seed}: best wrong={seed_best['wrong']} loss={seed_best['loss']:.6g} epoch={seed_best['epoch']}")

    # Return the best state for diagnostics, but mark as not ok so it is not submitted.
    if best_state is not None:
        best_model = BottleneckLocal3x3(hidden_channels=hidden_channels)
        best_model.load_state_dict(best_state)
        best_model.eval()
    else:
        best_model = None

    return best_model, {
        "ok": False,
        "task_id": task_id,
        "trainer": "bottleneck_local3x3_h4_cnn",
        "model_version": MODEL_VERSION,
        "hidden_channels": hidden_channels,
        "reason": "H=4 bottleneck did not reach exact visible match",
        "best_visible_wrong": best_wrong,
        "best_loss": best_loss,
        "attempts": attempts,
    }


def export_torch_model_to_onnx(model, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model.eval()
    dummy = torch.zeros((BATCH, CH, GRID_H, GRID_W), dtype=torch.float32)
    with torch.no_grad():
        torch.onnx.export(
            model,
            dummy,
            str(path),
            input_names=["input"],
            output_names=["output"],
            opset_version=10,
            do_constant_folding=True,
        )
    onnx.checker.check_model(str(path))
    return path


def train_family_task_h4(task, task_id):
    model, info = train_h4_bottleneck_for_task(task, task_id=task_id)
    if not info.get("ok"):
        return None, info
    return model, info


In [ ]:
# Build one H=4 bottleneck ONNX model per selected local_3x3 task.

# Clear stale models from earlier local runs.
for old_model_path in OUT_DIR.glob("task*.onnx"):
    old_model_path.unlink()

rows = []
for task_id in task_ids:
    print("\n" + "=" * 80)
    print("training", task_id)
    task = load_task(DATA_DIR, task_id)
    model, info = train_family_task_h4(task, task_id)

    row = {"task_id": task_id, **info}
    if model is not None and info.get("ok"):
        path = OUT_DIR / f"{task_id}.onnx"
        export_torch_model_to_onnx(model, path)

        # Validate exported ONNX, not just the PyTorch model.
        summary = visible_validation_summary(path, task)
        row.update({
            "saved": summary["wrong"] == 0,
            "path": str(path),
            "onnx_visible_right": summary["right"],
            "onnx_visible_wrong": summary["wrong"],
            "onnx_visible_total": summary["total"],
        })
        if summary["wrong"] != 0:
            path.unlink(missing_ok=True)
            row["saved"] = False
            row["reason"] = "PyTorch fit exact, but exported ONNX failed exact visible validation"
    else:
        row["saved"] = False

    rows.append(row)
    print({k: row.get(k) for k in ["task_id", "saved", "visible_wrong", "onnx_visible_wrong", "reason"]})

result_df = pd.DataFrame(rows)
display(result_df)

saved_count = int(result_df["saved"].sum()) if len(result_df) else 0
print("selected local_3x3 tasks:", len(task_ids))
print("models saved:", saved_count)

# Hard gate: do not create a misleading low-score submission.
assert saved_count == len(task_ids), f"H=4 bottleneck only saved {saved_count}/{len(task_ids)} models; not safe to submit."
assert (result_df["onnx_visible_wrong"].fillna(1).astype(int) == 0).all(), "At least one saved ONNX model is not exact."


In [ ]:
# Validate saved ONNX models again on visible examples, split by task.

validate_rows = []
for row in rows:
    if not row.get("saved"):
        continue
    task = load_task(DATA_DIR, row["task_id"])
    summary = visible_validation_summary(row["path"], task)
    validate_rows.append({
        "task_id": row["task_id"],
        "right": summary["right"],
        "wrong": summary["wrong"],
        "total": summary["total"],
    })

validate_df = pd.DataFrame(validate_rows)
display(validate_df)
assert len(validate_df) == len(task_ids)
assert (validate_df["wrong"].astype(int) == 0).all()


In [ ]:
# Architecture, memory, and estimated competition-cost report.

report_rows = []
for row in rows:
    if not row.get("saved"):
        continue
    task = load_task(DATA_DIR, row["task_id"])
    report = model_report(row["path"], task=task)
    arch = report["architecture"]
    mem = report["memory_profile"]
    perf = report["performance"]

    report_rows.append({
        "task_id": row["task_id"],
        "model_version": MODEL_VERSION,
        "hidden_channels": HIDDEN_CHANNELS,
        "file_size_bytes": arch.get("file_size_bytes"),
        "params": arch.get("params"),
        "nodes": arch.get("nodes"),
        "op_counts": json.dumps(arch.get("op_counts", {}), sort_keys=True),
        "static_memory_bytes": mem.get("static_memory_bytes"),
        "runtime_memory_bytes": mem.get("runtime_memory_bytes"),
        "estimated_cost_static": arch.get("params") + mem.get("static_memory_bytes"),
        "estimated_cost_runtime": arch.get("params") + mem.get("runtime_memory_bytes"),
        "train_right": perf["train"]["right"],
        "train_total": perf["train"]["total"],
        "train_accuracy": perf["train"]["accuracy"],
        "test_right": perf["test"]["right"],
        "test_total": perf["test"]["total"],
        "test_accuracy": perf["test"]["accuracy"],
        "arc_gen_right": perf["arc_gen"]["right"],
        "arc_gen_total": perf["arc_gen"]["total"],
        "arc_gen_accuracy": perf["arc_gen"]["accuracy"],
        "visible_right": perf["visible_all"]["right"],
        "visible_total": perf["visible_all"]["total"],
        "visible_accuracy": perf["visible_all"]["accuracy"],
    })

profile_df = pd.DataFrame(report_rows)
display(profile_df)
display(profile_df[["params", "static_memory_bytes", "runtime_memory_bytes", "estimated_cost_static", "estimated_cost_runtime"]].describe())

assert len(profile_df) == len(task_ids)
assert (profile_df["visible_accuracy"] == 1.0).all()


In [ ]:
# Create submission.zip only after all gates pass.

zip_path = create_submission_zip(OUT_DIR)
submission_zip = Path("/kaggle/working/submission.zip") if Path("/kaggle/working").exists() else Path.cwd() / "submission.zip"
shutil.copy2(zip_path, submission_zip)

manifest = {
    "family": FAMILY,
    "model_version": MODEL_VERSION,
    "hidden_channels": HIDDEN_CHANNELS,
    "task_count": len(task_ids),
    "saved_count": saved_count,
    "out_dir": str(OUT_DIR),
    "submission_zip": str(submission_zip),
}
manifest_path = OUT_DIR / f"{FAMILY}_{MODEL_VERSION}_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

profile_path = OUT_DIR / f"{FAMILY}_{MODEL_VERSION}_profile.csv"
profile_df.to_csv(profile_path, index=False)

print("family zip:", zip_path)
print("kaggle submission zip:", submission_zip)
print("manifest:", manifest_path)
print("profile:", profile_path)
manifest
